# Choosing box size: alpha-gamma sweep

Map how solution density, liftability to DNS, and TW vs equilibrium existence vary with (alpha, gamma) or equivalently (Lx, Lz).


## 1. Setup

In [1]:
using LinearAlgebra, Statistics, Random, SparseArrays
using DataFrames, Plots
using Dates
using Serialization
using Base.Threads

using CloudAtlas
using ChannelflowWrapper
using DifferentialEquations

# Physical parameters
Re = 400.0

# Symmetry subgroup(s) for EQB search
sx, sy, sz, tx, tz = halfbox_symmetries()
H_eqb = [sx * sy * sz, sz * tx * tz]

# Symmetry subgroup for TW1 search
H_tw = [(sx * sy) * (tx * tz)]

# Truncation
J, K, L = 2, 4, 5

# Sweep grid in Lx, Lz (more interpretable)
Lx_vals = range(2 * pi, 6 * pi; length=9)
Lz_vals = range(pi, 4 * pi; length=8)

# Convert to (alpha, gamma)
alpha_vals = 2 * pi ./ Lx_vals
gamma_vals = 2 * pi ./ Lz_vals

# ODE solver parameters
hookparams = SearchParams(ftol=1e-8, xtol=1e-12, Nnewton=30, Nhook=8, verbosity=0)

# Fuzzing / distinctness parameters
xnorm = 0.40
norm_threshold = 1e-2
distinct_tol = 1e-4

# Survival experiment parameters
survival_N_ic = 20
survival_E0 = 1e-2
survival_T_end = 200.0
survival_dt = 1.0
survival_E_thr_ratio = 1e-3

do_survival = true

# DNS lift settings (optional)
do_dns_lift_eqb = false
do_dns_lift_tw = false

base_dir = "notebooks/boxsize/boxsize_sweep_results"
mkpath(base_dir)

# EQB lift settings
symm_file_eqb = "notebooks/boxsize/sztx.asc"
reference_path_eqb = "./reference_field.nc"  # set to a valid DNS field
T_dns_eqb = 10.0

# TW1 lift settings
symm_file_tw = "notebooks/boxsize/sxytxz.asc"
reference_path_tw = "TW1-2pi1piRe200-40x49x40.nc"  # update if needed
T_dns_tw = 10.0


10.0

## 2. Define a standardized discovery protocol

This is a placeholder scaffold. Plug in your solver (Newton, GMRES, etc.) and
return a structured result with counts, solutions, and timings.


In [4]:
struct DiscoverResult
    n_converged::Int
    solutions::Vector{Any}
    representatives::Vector{Any}
    dns_attempts::Int
    dns_successes::Int
    timings::Dict{Symbol, Float64}
end

struct SolutionFingerprint
    cx::Float64
    cz::Float64
    nm::Float64
end

function is_distinct(new_fp::SolutionFingerprint, archive::Vector{SolutionFingerprint}; tol=1e-4)
    for fp in archive
        if isapprox(new_fp.cx, fp.cx, atol=tol) &&
           isapprox(new_fp.cz, fp.cz, atol=tol) &&
           isapprox(new_fp.nm, fp.nm, atol=tol)
            return false
        end
    end
    return true
end

# Distance in coefficient space: (x - y)' B (x - y)
function b_distance(x, y, B)
    dx = x - y
    dot(dx, B * dx)
end

function lift_eqb_to_dns(x, model, reference_field_converted, sol_dir; Re, symm_file, T)
    mkpath(sol_dir)
    guess_path = joinpath(sol_dir, "u_guess.nc")

    coeff2field(x, model.ijkl, reference_field_converted, guess_path)

    try
        findsoln(guess_path;
            R = Re, eqb = true,
            symms = abspath(symm_file),
            od = sol_dir, T = T
        )
        return true
    catch e
        println("findsoln failed: $e")
        return false
    end
end

function lift_tw_to_dns(x, cx, cz, model, reference_field_converted, sol_dir; Re, symm_file, T)
    mkpath(sol_dir)
    guess_path = joinpath(sol_dir, "u_guess.nc")
    sigma_file = joinpath(sol_dir, "sigma.asc")

    coeff2field(x, model.ijkl, reference_field_converted, guess_path)
    save_sigma(model, cx, cz, T, sigma_file)

    try
        findsoln(guess_path;
            R = Re, eqb = true,
            xrel = model.keep_cx, zrel = model.keep_cz,
            symms = abspath(symm_file),
            sigma = sigma_file, od = sol_dir, T = T
        )
        return true
    catch e
        println("findsoln failed: $e")
        return false
    end
end

function discover_eqb_solutions(; alpha, gamma, J, K, L, H, Re,
                                  Nguess=10_000, seed=0,
                                  xnorm=xnorm, norm_threshold=norm_threshold,
                                  distinct_tol=distinct_tol,
                                  hookparams=hookparams,
                                  do_dns_lift=do_dns_lift_eqb,
                                  base_dir=base_dir,
                                  reference_path=reference_path_eqb,
                                  symm_file=symm_file_eqb,
                                  T=T_dns_eqb)
    Random.seed!(seed)

    t0 = time()

    model = ODEModel(alpha, gamma, J, K, L, H)
    m = length(model)
    B = model.B

    f(x) = model.f(x, Re)
    Df(x) = model.Df(x, Re)

    io_lock = ReentrantLock()
    data_lock = ReentrantLock()

    solutions = Vector{Any}()
    representatives = Vector{Any}()

    dns_attempts = Atomic{Int}(0)
    dns_successes = Atomic{Int}(0)

    reference_field_converted = nothing
    if do_dns_lift
        mkpath(base_dir)
        if reference_path == "./reference_field.nc"
            lock(io_lock) do
                println("reference_path_eqb not set; skipping DNS lift")
            end
        else
            reference_field_converted = joinpath(base_dir, "reference_field_eqb_$(alpha)_$(gamma).nc")
            changegrid(reference_path, reference_field_converted; al=alpha, ga=gamma)
        end
    end

    function is_distinct_eqb(x)
        for r in representatives
            if b_distance(x, r[:x], B) < distinct_tol
                return false
            end
        end
        return true
    end

    @threads for i in 1:Nguess
        x_guess = randn(m)
        x_guess = xnorm / norm(x_guess) * x_guess

        x_star, converged = hookstepsolve(f, Df, x_guess, hookparams)
        if !converged
            continue
        end
        println("Found a convergent EQB solution!")

        solution_norm = norm(x_star)
        if solution_norm <= norm_threshold
            continue
        end

        residual = norm(f(x_star))
        sol = (x=x_star, norm=solution_norm, residual=residual)

        is_new = false
        lock(data_lock) do
            push!(solutions, sol)
            if is_distinct_eqb(x_star)
                push!(representatives, sol)
                is_new = true
            end
        end

        if is_new && do_dns_lift && reference_field_converted !== nothing
            timestamp = Dates.format(now(), "MM-DD-HHMMSS")
            sol_dir = joinpath(base_dir, "eqb_$(J)_$(K)_$(L)_id$(i)_$(timestamp)")

            atomic_add!(dns_attempts, 1)
            ok = lift_eqb_to_dns(x_star, model, reference_field_converted, sol_dir;
                                 Re=Re, symm_file=symm_file, T=T)
            if ok
                atomic_add!(dns_successes, 1)
            end
        end
    end

    timings = Dict{Symbol, Float64}(:elapsed => time() - t0)
    DiscoverResult(length(representatives), solutions, representatives,
                   dns_attempts[], dns_successes[], timings)
end

function discover_tw_solutions(; alpha, gamma, J, K, L, H, Re,
                                 Nguess=10_000, seed=0,
                                 xnorm=xnorm, norm_threshold=norm_threshold,
                                 hookparams=hookparams,
                                 do_dns_lift=do_dns_lift_tw,
                                 base_dir=base_dir,
                                 reference_path=reference_path_tw,
                                 symm_file=symm_file_tw,
                                 T=T_dns_tw)
    Random.seed!(seed)

    t0 = time()

    model = TWModel(alpha, gamma, J, K, L, H; normalize=false)
    m = length(model)

    io_lock = ReentrantLock()
    data_lock = ReentrantLock()

    solutions = Vector{Any}()
    representatives = Vector{Any}()
    solution_archive = Vector{SolutionFingerprint}()

    dns_attempts = Atomic{Int}(0)
    dns_successes = Atomic{Int}(0)

    reference_field_converted = nothing
    if do_dns_lift
        mkpath(base_dir)
        if reference_path == "TW1-2pi1piRe200-40x49x40.nc"
            lock(io_lock) do
                println("reference_path_tw not set; skipping DNS lift")
            end
        else
            reference_field_converted = joinpath(base_dir, "reference_field_tw_$(alpha)_$(gamma).nc")
            changegrid(reference_path, reference_field_converted; al=alpha, ga=gamma)
        end
    end

    @threads for i in 1:Nguess
        x_guess = randn(m)
        x_guess = xnorm / norm(x_guess) * x_guess
        cx_guess = randn() * 0.1
        cz_guess = randn() * 0.1
        ξ_guess = [x_guess; cx_guess; cz_guess]

        f(ξ) = model.g(ξ, Re)
        Df(ξ) = model.Dg(ξ, Re)

        ξ_star, converged = hookstepsolve(f, Df, ξ_guess, hookparams)
        if !converged
            continue
        end
        println("Found a convergent TW solution!")

        solution_norm = norm(ξ_star[1:m])
        if solution_norm <= norm_threshold
            continue
        end

        x_found, cx_found, cz_found = extract_components(ξ_star, model)
        new_fp = SolutionFingerprint(cx_found, cz_found, solution_norm)

        is_new = false
        lock(data_lock) do
            if is_distinct(new_fp, solution_archive)
                push!(solution_archive, new_fp)
                push!(representatives, (x=x_found, cx=cx_found, cz=cz_found, norm=solution_norm))
                is_new = true
            end
            push!(solutions, (x=x_found, cx=cx_found, cz=cz_found, norm=solution_norm))
        end

        if is_new && do_dns_lift && reference_field_converted !== nothing
            timestamp = Dates.format(now(), "MM-DD-HHMMSS")
            sol_dir = joinpath(base_dir, "tw_$(J)_$(K)_$(L)_id$(i)_$(timestamp)")

            atomic_add!(dns_attempts, 1)
            ok = lift_tw_to_dns(x_found, cx_found, cz_found, model, reference_field_converted, sol_dir;
                                Re=Re, symm_file=symm_file, T=T)
            if ok
                atomic_add!(dns_successes, 1)
            end
        end
    end

    timings = Dict{Symbol, Float64}(:elapsed => time() - t0)
    DiscoverResult(length(representatives), solutions, representatives,
                   dns_attempts[], dns_successes[], timings)
end

function survival_experiment(; alpha, gamma, J, K, L, H, Re,
                               N_ic=survival_N_ic, E0=survival_E0,
                               T_end=survival_T_end, save_dt=survival_dt,
                               E_thr_ratio=survival_E_thr_ratio,
                               seed=0)
    Random.seed!(seed)

    model = ODEModel(alpha, gamma, J, K, L, H)
    m = length(model)
    B = model.B

    E_thr = E0 * E_thr_ratio
    decay_times = Float64[]
    survivors = 0

    f!(dx, x, p, t) = (dx .= model.f(x, Re))

    for i in 1:N_ic
        x0 = randn(m)
        scale = sqrt((2 * E0) / dot(x0, B * x0))
        x0 .= scale .* x0

        prob = ODEProblem(f!, x0, (0.0, T_end))
        sol = solve(prob, Tsit5(); saveat=save_dt, abstol=1e-8, reltol=1e-6)

        energies = [0.5 * dot(u, B * u) for u in sol.u]
        t_decay = T_end
        for (t, E) in zip(sol.t, energies)
            if E < E_thr
                t_decay = t
                break
            end
        end

        if t_decay >= T_end
            survivors += 1
        end
        push!(decay_times, t_decay)
    end

    survival_fraction = survivors / N_ic
    median_decay_time = median(decay_times)
    return survival_fraction, median_decay_time
end

function discover_solutions(; alpha, gamma, J, K, L, H, Re,
                              Nguess=10_000, seed=0, mode=:eqb,
                              base_dir=base_dir)
    if mode == :eqb
        return discover_eqb_solutions(alpha=alpha, gamma=gamma, J=J, K=K, L=L, H=H, Re=Re,
                                      Nguess=Nguess, seed=seed, base_dir=base_dir)
    elseif mode == :tw
        return discover_tw_solutions(alpha=alpha, gamma=gamma, J=J, K=K, L=L, H=H, Re=Re,
                                     Nguess=Nguess, seed=seed, base_dir=base_dir)
    else
        error("unknown mode: $mode")
    end
end


discover_solutions (generic function with 1 method)

## 3. Clustering / uniqueness in coefficient space

Use the B inner-product metric to avoid counting duplicates.


In [5]:
function cluster_solutions(solutions, B; tol=1e-6)
    reps = Vector{Any}()
    for s in solutions
        x = s[:x]
        is_new = true
        for r in reps
            if b_distance(x, r[:x], B) < tol
                is_new = false
                break
            end
        end
        if is_new
            push!(reps, s)
        end
    end
    reps
end


cluster_solutions (generic function with 1 method)

## 4. Lift to DNS (optional)

Run DNS only on clustered representatives to keep cost down.


In [6]:
# DNS lift happens inside discover_eqb_solutions and discover_tw_solutions when enabled.


## 5. Sweep loop and collect metrics

For each (alpha, gamma) pair, discover equilibria/TWs, cluster them, and compute
rates per 10k guesses. Optionally lift selected reps to DNS.


In [ ]:
results = DataFrame(
    Lx=Float64[], Lz=Float64[], alpha=Float64[], gamma=Float64[],
    n_eqb=Int[], n_tw=Int[],
    eqb_per_10k=Float64[], tw_per_10k=Float64[],
    eqb_dns_success_rate=Float64[], tw_dns_success_rate=Float64[],
    survival_fraction=Float64[], median_decay_time=Float64[],
)

Nguess = 10_000

for (i, Lx) in enumerate(Lx_vals)
    alpha = alpha_vals[i]
    for (j, Lz) in enumerate(Lz_vals)
        gamma = gamma_vals[j]

        eqb_base_dir = joinpath(base_dir, "eqb_Lx$(round(Lx, digits=3))_Lz$(round(Lz, digits=3))")
        tw_base_dir = joinpath(base_dir, "tw_Lx$(round(Lx, digits=3))_Lz$(round(Lz, digits=3))")

        eqb = discover_solutions(alpha=alpha, gamma=gamma, J=J, K=K, L=L, H=H_eqb, Re=Re,
                                 Nguess=Nguess, mode=:eqb, base_dir=eqb_base_dir)

        tw = discover_solutions(alpha=alpha, gamma=gamma, J=J, K=K, L=L, H=H_tw, Re=Re,
                                Nguess=Nguess, mode=:tw, base_dir=tw_base_dir)

        n_eqb = eqb.n_converged
        n_tw = tw.n_converged

        eqb_dns_rate = eqb.dns_attempts > 0 ? (eqb.dns_successes / eqb.dns_attempts) : NaN
        tw_dns_rate = tw.dns_attempts > 0 ? (tw.dns_successes / tw.dns_attempts) : NaN

        if do_survival
            surv_frac, med_decay = survival_experiment(alpha=alpha, gamma=gamma, J=J, K=K, L=L,
                                                       H=H_eqb, Re=Re)
        else
            surv_frac, med_decay = NaN, NaN
        end

        push!(results, (
            Lx, Lz, alpha, gamma,
            n_eqb, n_tw,
            10_000 * n_eqb / Nguess,
            10_000 * n_tw / Nguess,
            eqb_dns_rate, tw_dns_rate,
            surv_frac, med_decay,
        ))

        serialize(joinpath(base_dir, "eqb_results_Lx$(round(Lx, digits=3))_Lz$(round(Lz, digits=3)).jls"), eqb)
        serialize(joinpath(base_dir, "tw_results_Lx$(round(Lx, digits=3))_Lz$(round(Lz, digits=3)).jls"), tw)
    end
end

serialize(joinpath(base_dir, "boxsize_sweep_results.jls"), results)

results[1:10, :]


J,K,L,m == 2,4,5,124
(2J+1)(2K+1)(2L+1) + 1 == 496
Making matrices B,A1,A2,S3...
Making quadratic operator N...
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 
Found a convergent EQB solution!
Found a convergent EQB solution!
Found a convergent EQB solution!
Found a convergent EQB solution!
Found a convergent EQB solution!
Found a convergent EQB solution!
Found a convergent EQB solution!
Found a convergent EQB solution!
Found a convergent EQB solution!
Found a convergent EQB solution!
Found a convergent EQB solution!
Found a convergent EQB solution!
Found a convergent EQB solution!
Found a convergent EQB solution!
Found a convergent EQB solution!
Foun

## 6. Heatmaps / phase diagrams

Make density maps and liftability overlays for the sweep results.


In [ ]:
function heatmap_from_results(results, field; title="", xlabel="Lz", ylabel="Lx")
    Lx = sort(unique(results.Lx))
    Lz = sort(unique(results.Lz))
    grid = fill(NaN, length(Lx), length(Lz))
    for row in eachrow(results)
        ix = findfirst(==(row.Lx), Lx)
        iz = findfirst(==(row.Lz), Lz)
        grid[ix, iz] = row[field]
    end
    plt = heatmap(Lz, Lx, grid; xlabel=xlabel, ylabel=ylabel, title=title)
    return plt, Lx, Lz, grid
end

plot_title = "Re=$(Re), J=$(J), K=$(K), L=$(L)"

# A1: survival fraction heatmap
plt1, Lx_grid, Lz_grid, Sgrid = heatmap_from_results(
    results, :survival_fraction,
    title="Survival fraction (" * plot_title * ")",
    xlabel="Lz", ylabel="Lx",
)
contour!(Lz_grid, Lx_grid, Sgrid; levels=[0.1, 0.5, 0.9], color=:white, linewidth=1)

# A2: median decay time heatmap
plt2, Lx_grid2, Lz_grid2, Tgrid = heatmap_from_results(
    results, :median_decay_time,
    title="Median decay time (" * plot_title * ")",
    xlabel="Lz", ylabel="Lx",
)


In [ ]:
plt_tw, _, _, _ = heatmap_from_results(
    results, :tw_per_10k,
    title="TWs per 10k guesses (" * plot_title * ")",
    xlabel="Lz", ylabel="Lx",
)


In [ ]:
plt_dns, _, _, _ = heatmap_from_results(
    results, :tw_dns_success_rate,
    title="TW DNS lift success rate (" * plot_title * ")",
    xlabel="Lz", ylabel="Lx",
)


## 7. Minimum box sustaining turbulence proxy

Two options:

- ODE chaos proxy: long integrations from random ICs and detect chaotic behavior.
- ECS richness proxy: count distinct EQ/TW across the grid and look for onset.


In [ ]:
# Placeholder: ODE chaos proxy
function chaos_proxy(alpha, gamma)
    # TODO: integrate and estimate Lyapunov or broadband spectrum
    return NaN
end


In [ ]:
# Example: build a proxy grid
# chaos_grid = [chaos_proxy(alpha_vals[i], gamma_vals[j]) for i in eachindex(alpha_vals), j in eachindex(gamma_vals)]
# heatmap(Lz_vals, Lx_vals, chaos_grid, xlabel="Lz", ylabel="Lx", title="Chaos proxy")


## 8. Identify promising boxes

Filter for boxes with high ODE density and high DNS success (if available).


In [ ]:
# C1: Pareto scatter for box selection
valid = filter(row -> isfinite(row.tw_per_10k) && isfinite(row.tw_dns_success_rate) && isfinite(row.survival_fraction), results)

score(row) = row.survival_fraction * row.tw_per_10k * row.tw_dns_success_rate
sorted = sort(valid, :tw_per_10k, rev=true)

plt_pareto = scatter(
    valid.tw_per_10k,
    valid.tw_dns_success_rate;
    markersize=10 .* valid.survival_fraction,
    xlabel="TW distinct per 10k",
    ylabel="TW DNS lift success rate",
    title="Pareto: density vs liftability (" * plot_title * ")",
    label=false,
)

# Annotate top 5 by score
if nrow(valid) > 0
    top = sort(valid, :tw_per_10k, rev=true)
    top = sort(top, :tw_dns_success_rate, rev=true)
    ranked = sort(valid, by=score, rev=true)
    for row in eachrow(ranked[1:min(5, nrow(ranked)), :])
        annotate!(row.tw_per_10k, row.tw_dns_success_rate,
                  text("Lx=$(round(row.Lx, digits=2)), Lz=$(round(row.Lz, digits=2))", :left, 8))
    end
end

plt_pareto
